In [1]:
import os
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession

# Uruchomienie lokalnego klastra Spark (w kontenerze)
spark = SparkSession.builder.appName("IcebergDemo").getOrCreate()
print("Środowisko gotowe! Spark uruchomiony.")

Środowisko gotowe! Spark uruchomiony.


26/09/29 11:01:11 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
# Generujemy 3 miliony wierszy "na sucho" w Pythonie (Pandas)
print("Generowanie 3 milionów rekordów...")
df_pd = pd.DataFrame({
    'id': np.arange(3000000),
    'name': ['Jan'] * 3000000,
    'city': ['Warszawa'] * 3000000,
    'salary': np.random.randint(4000, 15000, size=3000000)
})
print("Zrobione!")

Generowanie 3 milionów rekordów...
Zrobione!


In [3]:
# Zapisujemy jako staroświecki CSV i nowoczesny Parquet
csv_path = '/home/iceberg/warehouse/test.csv'
parquet_path = '/home/iceberg/warehouse/test.parquet'

print("Zapisuję do CSV (to chwilę potrwa)...")
df_pd.to_csv(csv_path, index=False)

print("Zapisuję do Parquet...")
df_pd.to_parquet(parquet_path, index=False)

# Porównujemy rozmiary
csv_size = os.path.getsize(csv_path) / (1024 * 1024)
parquet_size = os.path.getsize(parquet_path) / (1024 * 1024)

print(f"\n--- WYNIKI ---")
print(f"Rozmiar CSV:     {csv_size:.2f} MB")
print(f"Rozmiar Parquet: {parquet_size:.2f} MB")
print(f"Parquet jest {(csv_size/parquet_size):.1f} razy mniejszy!")

Zapisuję do CSV (to chwilę potrwa)...
Zapisuję do Parquet...

--- WYNIKI ---
Rozmiar CSV:     74.63 MB
Rozmiar Parquet: 17.39 MB
Parquet jest 4.3 razy mniejszy!


In [4]:
# Ładujemy nasz zoptymalizowany plik Parquet do Sparka
df_spark = spark.read.parquet(parquet_path)

# Tworzymy z niego prawdziwą tabelę Iceberg (katalog jest już skonfigurowany w tym dockerze)
print("Tworzę tabelę Iceberg...")
df_spark.writeTo("demo.default.employees").createOrReplace()
print("Tabela gotowa!")

# Pokażmy, ile jest wierszy:
spark.sql("SELECT COUNT(*) as Liczba_Pracownikow FROM demo.default.employees").show()

Tworzę tabelę Iceberg...


Py4JJavaError: An error occurred while calling o52.createOrReplace.
: org.apache.iceberg.exceptions.RESTException: Error occurred while processing GET request
	at org.apache.iceberg.rest.HTTPClient.execute(HTTPClient.java:327)
	at org.apache.iceberg.rest.BaseHTTPClient.get(BaseHTTPClient.java:77)
	at org.apache.iceberg.rest.RESTSessionCatalog.fetchConfig(RESTSessionCatalog.java:1089)
	at org.apache.iceberg.rest.RESTSessionCatalog.initialize(RESTSessionCatalog.java:261)
	at org.apache.iceberg.rest.RESTCatalog.initialize(RESTCatalog.java:78)
	at org.apache.iceberg.CatalogUtil.loadCatalog(CatalogUtil.java:274)
	at org.apache.iceberg.CatalogUtil.buildIcebergCatalog(CatalogUtil.java:328)
	at org.apache.iceberg.spark.SparkCatalog.buildIcebergCatalog(SparkCatalog.java:153)
	at org.apache.iceberg.spark.SparkCatalog.initialize(SparkCatalog.java:752)
	at org.apache.spark.sql.connector.catalog.Catalogs$.load(Catalogs.scala:65)
	at org.apache.spark.sql.connector.catalog.CatalogManager.$anonfun$cat